In [9]:
import numpy as np
import random

In [10]:
"""Creates a matrix with the given number of rows and columns.
0: empty
1: dust
2: obstacle
"""
rows = int(input("Enter the number of rows: "))
cols = int(input("Enter the number of columns: "))

house = np.random.randint(0, 3, size=(rows, cols))

house[rows-1][cols-1] = 3  # Ensure the bottom-right corner is dock
house[0][0] = 0  # Ensure the top-left corner is empty
robot_pos = [0, 0]  # Starting position of the robot
min_steps_to_clean = rows * cols  # Minimum steps to clean the house

print("Initial House State: \n" + str(house))

# Define the 8 possible movement directions (including diagonals)
directions = {
    "UP": (-1, 0), "DOWN": (1, 0), "LEFT": (0, -1), "RIGHT": (0, 1),
    "UP_LEFT": (-1, -1), "UP_RIGHT": (-1, 1), 
    "DOWN_LEFT": (1, -1), "DOWN_RIGHT": (1, 1)
}

Initial House State: 
[[0 1 1 1 1 1 2 2 1 2]
 [1 2 2 1 2 2 1 0 2 0]
 [0 0 2 0 2 2 0 1 2 1]
 [1 2 0 0 1 2 1 1 1 1]
 [0 0 1 1 0 1 1 2 2 1]
 [1 0 0 0 2 1 1 0 2 0]
 [2 0 1 0 0 0 0 1 1 1]
 [0 2 0 0 0 0 2 0 0 2]
 [2 1 2 0 0 0 0 2 0 0]
 [2 2 0 0 1 2 1 2 0 3]]


In [11]:
def choose_action(vision, dock_active):
    """
    vision: 3x3 view
    Priority: STOP > SUCK > GO_TO_DIRT > GO_TO_DOCK > RANDOM_MOVE
    """
    # Stop if on dock
    if vision[1, 1] == 3 and dock_active:
        return "STOP"
    
    # Suck if on dust
    if vision[1, 1] == 1:
        return "SUCK"

    # Find dirty spots in the vision
    dirty_spots = []
    for name, (dr, dc) in directions.items():
        if vision[1 + dr, 1 + dc] == 1:
            dirty_spots.append(name)
    
    if dirty_spots:
        return random.choice(dirty_spots) # Move towards a random dirty spot

    # Look for the dock in the vision
    if dock_active:
        for name, (dr, dc) in directions.items():
            if vision[1 + dr, 1 + dc] == 3:
                return name # Move towards the dock

    # If no dust or dock, move randomly to a safe direction (not an obstacle)
    safe_moves = [name for name, (dr, dc) in directions.items() if vision[1 + dr, 1 + dc] != 2 and vision[1 + dr, 1 + dc] != 3]
            
    return random.choice(safe_moves) if safe_moves else "STAY"

In [12]:
def get_local_vision(pos, grid):
    r, c = pos
    # Padding the grid to handle edge cases easily
    padded = np.pad(grid, pad_width=1, mode='constant', constant_values=2)
    return padded[r:r+3, c:c+3]

steps = 0
max_steps = rows * cols * 10 # battery limit to prevent infinite loops

while True:
    steps += 1

    dock_active = (steps >= min_steps_to_clean) # Activate dock after minimum cleaning steps

    vision = get_local_vision(robot_pos, house)
    
    # Decision
    action = choose_action(vision, dock_active)
    
    print(f"Bước {steps}: Tại {robot_pos}, quyết định: {action}")

    if action == "STOP":
        print(">>> ROBOT ĐÃ LEO LÊN DOCK. NHIỆM VỤ HOÀN THÀNH!")
        break

    if action == "SUCK":
        house[robot_pos[0], robot_pos[1]] = 0
    elif action == "STAY":
        print("--- BỊ KẸT ---")
        break
    else:
        # Thực hiện di chuyển theo 8 hướng
        offset = directions[action]
        robot_pos[0] += offset[0]
        robot_pos[1] += offset[1]

print("\nBản đồ cuối cùng:\n", house)

Bước 1: Tại [0, 0], quyết định: RIGHT
Bước 2: Tại [0, 1], quyết định: SUCK
Bước 3: Tại [0, 1], quyết định: DOWN_LEFT
Bước 4: Tại [1, 0], quyết định: SUCK
Bước 5: Tại [1, 0], quyết định: UP_RIGHT
Bước 6: Tại [0, 1], quyết định: RIGHT
Bước 7: Tại [0, 2], quyết định: SUCK
Bước 8: Tại [0, 2], quyết định: DOWN_RIGHT
Bước 9: Tại [1, 3], quyết định: SUCK
Bước 10: Tại [1, 3], quyết định: UP_RIGHT
Bước 11: Tại [0, 4], quyết định: SUCK
Bước 12: Tại [0, 4], quyết định: LEFT
Bước 13: Tại [0, 3], quyết định: SUCK
Bước 14: Tại [0, 3], quyết định: DOWN
Bước 15: Tại [1, 3], quyết định: DOWN
Bước 16: Tại [2, 3], quyết định: DOWN_RIGHT
Bước 17: Tại [3, 4], quyết định: SUCK
Bước 18: Tại [3, 4], quyết định: DOWN_LEFT
Bước 19: Tại [4, 3], quyết định: SUCK
Bước 20: Tại [4, 3], quyết định: LEFT
Bước 21: Tại [4, 2], quyết định: SUCK
Bước 22: Tại [4, 2], quyết định: DOWN_RIGHT
Bước 23: Tại [5, 3], quyết định: DOWN_LEFT
Bước 24: Tại [6, 2], quyết định: SUCK
Bước 25: Tại [6, 2], quyết định: UP_RIGHT
Bước 26: Tại